# First (Benchmark) Model: Gradient Boost
# Refined Model: RNN w/ LSTM

In [46]:
# Import necssary libraries
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from tensorflow.keras.utils import to_categorical
import numpy as np
import datetime
import pandas as pd

In [92]:
# Import data
#prod_data = pd.read_csv
training_data = pd.read_csv(r'C:\Users\theni\Downloads\training_flight_data.csv')
test_data = pd.read_csv(r'C:\Users\theni\Downloads\valid_flight_data.csv')
valid_data = pd.read_csv(r'C:\Users\theni\Downloads\test_flight_data.csv')

In [93]:
training_data.columns = ['sample_id', 'time_step', 'aileron_pos_lh_deg', 'aileron_pos_rh_deg', 'corrected_angle_of_attack_deg', 'baro_correct_alt_lsp_ft', 'computed_airspeed_lsp_knots',
    'selected_course_deg', 'drift_angle_deg', 'elevator_pos_left_deg', 'te_flap_pos_disc', 'glideslope_dev_perc', 'selected_heading_deg', 'localizer_dev_perc', 'core_speed_avg_perc',
    'total_pressure_lsp_millibar', 'pitch_angle_lsp_deg', 'roll_angle_lsp_deg', 'rudder_pos_deg', 'true_heading_lsp_deg', 'vertical_accel_g', 'wind_speed_knots', 'label']
test_data.columns = ['sample_id', 'time_step', 'aileron_pos_lh_deg', 'aileron_pos_rh_deg', 'corrected_angle_of_attack_deg', 'baro_correct_alt_lsp_ft', 'computed_airspeed_lsp_knots',
    'selected_course_deg', 'drift_angle_deg', 'elevator_pos_left_deg', 'te_flap_pos_disc', 'glideslope_dev_perc', 'selected_heading_deg', 'localizer_dev_perc', 'core_speed_avg_perc',
    'total_pressure_lsp_millibar', 'pitch_angle_lsp_deg', 'roll_angle_lsp_deg', 'rudder_pos_deg', 'true_heading_lsp_deg', 'vertical_accel_g', 'wind_speed_knots', 'label']
valid_data.columns = ['sample_id', 'time_step', 'aileron_pos_lh_deg', 'aileron_pos_rh_deg', 'corrected_angle_of_attack_deg', 'baro_correct_alt_lsp_ft', 'computed_airspeed_lsp_knots',
    'selected_course_deg', 'drift_angle_deg', 'elevator_pos_left_deg', 'te_flap_pos_disc', 'glideslope_dev_perc', 'selected_heading_deg', 'localizer_dev_perc', 'core_speed_avg_perc',
    'total_pressure_lsp_millibar', 'pitch_angle_lsp_deg', 'roll_angle_lsp_deg', 'rudder_pos_deg', 'true_heading_lsp_deg', 'vertical_accel_g', 'wind_speed_knots', 'label']

In [94]:
training_data.head(5)

,sample_id,time_step,aileron_pos_lh_deg,aileron_pos_rh_deg,corrected_angle_of_attack_deg,baro_correct_alt_lsp_ft,computed_airspeed_lsp_knots,selected_course_deg,drift_angle_deg,elevator_pos_left_deg,...,localizer_dev_perc,core_speed_avg_perc,total_pressure_lsp_millibar,pitch_angle_lsp_deg,roll_angle_lsp_deg,rudder_pos_deg,true_heading_lsp_deg,vertical_accel_g,wind_speed_knots,label
0,0,0,81.261190,82.652336,-8.111792,1969.6174,155.57140,-2.109358,-0.692778,-4.952854,...,0.006664,70.74118,985.42550,-3.662262,0.785912,-0.390141,-1.081870,0.972379,12.625183,0.0
1,0,1,79.604095,81.015700,-7.644611,1955.6995,154.51205,-2.109358,-0.867216,-5.198349,...,0.007448,70.71775,985.52030,-3.665276,0.046774,-0.756234,-0.704820,0.770077,11.893839,0.0
2,0,2,81.302110,80.770200,-7.552573,1940.0267,153.32867,-2.109358,-1.424093,-4.830105,...,0.009604,70.70276,985.41650,-3.940319,0.804820,-1.325632,-0.240446,0.543937,12.559112,0.0
3,0,3,82.345470,83.900276,-8.395265,1924.5493,150.88818,-2.109358,-1.141912,-4.625526,...,0.009408,70.74159,984.81710,-4.275129,1.077102,-0.326884,-0.191627,1.062817,10.542998,0.0
4,0,4,81.874930,82.754620,-7.854284,1905.3670,150.69461,-2.109358,-0.724660,-4.400490,...,0.009408,70.57045,985.23065,-4.241483,1.654806,0.129545,-0.528425,0.867628,9.713539,0.0


In [95]:
test_data.head(5)

,sample_id,time_step,aileron_pos_lh_deg,aileron_pos_rh_deg,corrected_angle_of_attack_deg,baro_correct_alt_lsp_ft,computed_airspeed_lsp_knots,selected_course_deg,drift_angle_deg,elevator_pos_left_deg,...,localizer_dev_perc,core_speed_avg_perc,total_pressure_lsp_millibar,pitch_angle_lsp_deg,roll_angle_lsp_deg,rudder_pos_deg,true_heading_lsp_deg,vertical_accel_g,wind_speed_knots,label
0,15,0,80.606540,81.54761,-5.420757,2468.7761,180.01007,34.98022,3.319528,-3.500336,...,-0.003136,61.523205,978.03235,-1.985861,-1.344085,-2.817326,24.699790,1.015660,14.561880,0.0
1,15,1,79.849594,80.42242,-5.116660,2453.4820,178.55055,34.98022,3.403981,-3.561710,...,-0.003332,61.480885,977.95966,-1.719461,-1.292599,1.381010,24.573366,1.040051,14.044608,0.0
2,15,2,81.506690,82.01814,-4.978041,2441.3076,179.56377,34.98022,3.030978,-3.459419,...,-0.002940,61.483383,978.70500,-1.735226,0.008662,-3.048076,24.782553,1.032340,14.027080,0.0
3,15,3,81.813560,82.79555,-5.357613,2426.1600,177.57582,34.98022,3.490517,-3.664001,...,-0.002156,61.506645,978.21350,-1.802503,1.149909,1.843847,24.539455,1.000205,13.811078,0.0
4,15,4,82.652336,83.10242,-6.084550,2411.2341,179.69500,34.98022,3.546878,-3.664001,...,-0.002940,61.506870,980.06050,-2.078131,0.974984,-2.606024,24.573948,0.967859,15.460607,0.0


In [96]:
valid_data.head(5)

,sample_id,time_step,aileron_pos_lh_deg,aileron_pos_rh_deg,corrected_angle_of_attack_deg,baro_correct_alt_lsp_ft,computed_airspeed_lsp_knots,selected_course_deg,drift_angle_deg,elevator_pos_left_deg,...,localizer_dev_perc,core_speed_avg_perc,total_pressure_lsp_millibar,pitch_angle_lsp_deg,roll_angle_lsp_deg,rudder_pos_deg,true_heading_lsp_deg,vertical_accel_g,wind_speed_knots,label
0,18,0,83.184250,83.67523,-6.792032,1612.6204,128.703580,-68.026855,-6.184055,-6.569035,...,-0.348292,82.513760,977.36970,0.429042,20.637896,-0.185368,-138.77364,0.565935,16.215496,0.0
1,18,1,82.897835,84.22760,-6.425116,1613.3372,126.980020,-68.026855,-6.091631,-6.303082,...,-0.343196,82.715600,976.66974,0.300326,19.733183,-0.274315,-137.08560,0.705978,15.904416,0.0
2,18,2,82.877380,84.18669,-5.949559,1610.4536,126.013230,-68.026855,-6.332006,-7.735142,...,-0.333004,82.386680,976.38824,-0.242279,17.884089,-1.447740,-133.74431,0.517630,15.956974,0.0
3,18,3,82.856920,83.98210,-6.675802,1607.0038,125.520970,-68.026855,-6.655682,-5.955296,...,-0.328692,82.649414,976.35790,-1.455954,15.929121,-1.553204,-132.06366,0.672479,15.911472,0.0
4,18,4,82.877380,84.10486,-5.960420,1601.6227,125.407166,-68.026855,-7.219427,-5.750717,...,-0.325556,82.341070,976.48470,-1.506611,14.173342,-1.781981,-128.92786,0.071948,16.908192,0.0


In [97]:
flight_metadata_df = pd.read_csv(r"C:\Users\theni\Downloads\DASHlink_full_fourclass_raw_meta (1).csv")
sample_ids = training_data['sample_id'].unique()
train_flight_metadata_train = flight_metadata_df[flight_metadata_df['data_instance'].isin(sample_ids)]

In [98]:
train_flight_metadata_train.head()

,data_instance,flight_record,departure_airport,departure_runway,arrival_airport,arrival_runway,label
0,0,652200101120916,KSGF,32,KMEM,36L,0
2,2,652200101121341,KMCI,19R,KMEM,36R,0
3,3,652200101130002,KMEM,18C,KPNS,17,0
5,5,652200101130931,KSTL,30L,KMEM,9,0
6,6,652200101132030,KMEM,36L,KSGF,14,0


In [99]:
arrival_and_samp = train_flight_metadata_train.copy()
arrival_and_samp = arrival_and_samp[['data_instance', 'arrival_airport']]
arrival_and_samp.columns = ['sample_id', 'arrival_airport']
arrival_and_samp['arrival_airport_catenc'], arrival_airport_mapping = pd.factorize(arrival_and_samp['arrival_airport'])
arrival_and_samp.drop(columns=['arrival_airport'], inplace=True)
arrival_and_samp['arrival_airport_catenc'] = arrival_and_samp['arrival_airport_catenc'].astype(float)
arrival_and_samp.head()

,sample_id,arrival_airport_catenc
0,0,0.0
2,2,0.0
3,3,1.0
5,5,0.0
6,6,2.0


In [101]:
training_data["record_identifier"] = training_data["sample_id"].astype(str) + "_" + df["time_step"].astype(str)

training_data["time_step"] = training_data["time_step"].astype(float)
training_data.tail()

,sample_id,time_step,aileron_pos_lh_deg,aileron_pos_rh_deg,corrected_angle_of_attack_deg,baro_correct_alt_lsp_ft,computed_airspeed_lsp_knots,selected_course_deg,drift_angle_deg,elevator_pos_left_deg,...,core_speed_avg_perc,total_pressure_lsp_millibar,pitch_angle_lsp_deg,roll_angle_lsp_deg,rudder_pos_deg,true_heading_lsp_deg,vertical_accel_g,wind_speed_knots,label,record_identifier
6389915,99835,155.0,88.114624,82.222720,6.874017,629.22797,105.47570,34.98022,-1.176979,-3.316216,...,63.319550,1017.93130,4.400115,0.204086,-0.666766,29.992224,1.082715,3.502982,3.0,99835_155.0
6389916,99835,156.0,81.956764,86.375694,6.319379,623.89624,104.67262,34.98022,-1.019591,1.552792,...,61.987160,1017.81550,4.322569,-0.970712,-0.622764,29.740330,1.058380,3.245408,3.0,99835_156.0
6389917,99835,157.0,86.396150,86.027910,4.170040,624.05475,101.82832,34.98022,-1.167051,1.961948,...,61.082344,1016.82410,3.697519,0.035246,-0.695083,29.869590,1.056430,3.285423,3.0,99835_157.0
6389918,99835,158.0,85.577835,85.270966,1.343325,624.10240,99.21132,34.98022,-0.159543,10.063316,...,58.991150,1015.96124,2.390611,0.591246,-0.648484,29.010420,0.966415,3.440334,3.0,99835_158.0
6389919,99835,159.0,84.636765,86.682560,1.255068,624.51654,97.39182,34.98022,0.457792,9.797363,...,56.885094,1015.51624,2.680404,0.820136,-0.642458,28.177433,1.043380,2.406222,3.0,99835_159.0


In [102]:
df = training_data
import random
num_unique = (int)(len((df[df['label'] == 3]))/160)
rand_zeroes = df[df['label'] == 0]['sample_id'].unique()
print(num_unique)
unique_zeros = random.choices(rand_zeroes, k=num_unique)
print(len(unique_zeros))
rand_ones = df[df['label'] == 1]['sample_id'].unique()
unique_ones = random.choices(rand_ones, k=num_unique)
rand_twos = df[df['label'] == 2]['sample_id'].unique()
unique_twos = random.choices(rand_twos, k=num_unique)
sampleids_to_keep = df[df['label'] == 3]['sample_id'].unique().tolist()
print(len(sampleids_to_keep))
sampleids_to_keep.extend(unique_ones)
print(len(sampleids_to_keep))
sampleids_to_keep.extend(unique_twos)
print(len(sampleids_to_keep))
sampleids_to_keep.extend(unique_zeros)
print(len(sampleids_to_keep))

training_df = df[df['sample_id'].isin(sampleids_to_keep)]

382
382
382
764
1146
1528


In [103]:
training_df = training_df.merge(arrival_and_samp, on='sample_id', how='left')

In [104]:
training_df.head(5)

,sample_id,time_step,aileron_pos_lh_deg,aileron_pos_rh_deg,corrected_angle_of_attack_deg,baro_correct_alt_lsp_ft,computed_airspeed_lsp_knots,selected_course_deg,drift_angle_deg,elevator_pos_left_deg,...,total_pressure_lsp_millibar,pitch_angle_lsp_deg,roll_angle_lsp_deg,rudder_pos_deg,true_heading_lsp_deg,vertical_accel_g,wind_speed_knots,label,record_identifier,arrival_airport_catenc
0,120,0.0,82.836460,83.879820,-5.907745,2590.2136,171.83261,-61.08354,0.274361,-4.584610,...,970.93940,-1.577371,1.854986,0.208909,-71.07937,1.059393,11.987747,0.0,120_0.0,9.0
1,120,1.0,82.181810,83.347910,-7.820658,2580.5974,170.97165,-61.08354,0.398227,-4.830105,...,970.88460,-1.776710,0.340545,-0.284346,-71.22791,1.018832,12.136392,0.0,120_1.0,9.0
2,120,2.0,81.527145,82.284090,-7.028162,2570.8286,169.00168,-61.08354,-0.132570,-5.771175,...,969.76276,-1.610647,-0.221400,-1.653273,-70.45259,1.093805,12.496269,0.0,120_2.0,9.0
3,120,3.0,81.793106,82.877380,-8.150268,2564.8638,165.28168,-61.08354,-0.085725,-6.712242,...,968.35620,-2.485330,0.391957,0.242288,-70.70134,1.029731,10.529162,0.0,120_3.0,9.0
4,120,4.0,81.670350,82.652336,-9.836048,2558.7120,164.48022,-61.08354,0.312927,-5.893921,...,968.04785,-3.635063,0.225370,-0.488405,-70.73580,0.884186,11.255740,0.0,120_4.0,9.0


In [105]:
float_columns = [col for col in training_df.columns if col not in ['sample_id', 'time_step', 'label'] and training_df[col].dtype == 'float64']


from sklearn.preprocessing import MinMaxScaler

df_scaled = training_df.copy()
scalers = []
for col in float_columns:
  new_scaler = MinMaxScaler()
  df_scaled[col] = new_scaler.fit_transform(df_scaled[[col]])
  scalers.append(new_scaler)

In [106]:
df_scaled.head()

,sample_id,time_step,aileron_pos_lh_deg,aileron_pos_rh_deg,corrected_angle_of_attack_deg,baro_correct_alt_lsp_ft,computed_airspeed_lsp_knots,selected_course_deg,drift_angle_deg,elevator_pos_left_deg,...,total_pressure_lsp_millibar,pitch_angle_lsp_deg,roll_angle_lsp_deg,rudder_pos_deg,true_heading_lsp_deg,vertical_accel_g,wind_speed_knots,label,record_identifier,arrival_airport_catenc
0,120,0.0,0.497266,0.510575,0.389917,0.214768,0.533458,0.332269,0.490072,0.452718,...,0.713788,0.563042,0.543024,0.536243,0.383416,0.890935,0.199952,0.0,120_0.0,0.097826
1,120,1.0,0.489314,0.504106,0.320045,0.213981,0.528777,0.332269,0.492713,0.449739,...,0.713653,0.554148,0.523608,0.531688,0.383140,0.883841,0.202352,0.0,120_1.0,0.097826
2,120,2.0,0.481362,0.491167,0.348992,0.213182,0.518066,0.332269,0.481393,0.438322,...,0.710909,0.561557,0.516404,0.519045,0.384578,0.896954,0.208163,0.0,120_2.0,0.097826
3,120,3.0,0.484593,0.498383,0.308006,0.212694,0.497841,0.332269,0.482392,0.426905,...,0.707467,0.522532,0.524267,0.536551,0.384117,0.885747,0.176400,0.0,120_3.0,0.097826
4,120,4.0,0.483101,0.495646,0.246430,0.212190,0.493483,0.332269,0.490894,0.436833,...,0.706712,0.471236,0.522132,0.529803,0.384053,0.860290,0.188132,0.0,120_4.0,0.097826


In [107]:
train_x = df_scaled.copy()
train_x = train_x.drop(columns=['label','sample_id'])
train_y = df_scaled['label']

In [108]:
valid_ids = valid_data['sample_id'].unique()
print(valid_ids)
flight_metadata_valid = flight_metadata_df[flight_metadata_df['data_instance'].isin(valid_ids)]
flight_metadata_valid.head()
arrival_and_samp = flight_metadata_valid.copy()
arrival_and_samp = arrival_and_samp[['data_instance', 'arrival_airport']]
arrival_and_samp.columns = ['sample_id', 'arrival_airport']
arrival_and_samp
valid_df = valid_data.merge(arrival_and_samp, on='sample_id', how='left')
valid_df['arrival_airport_catenc'] = valid_df['arrival_airport'].map(dict(enumerate(arrival_airport_mapping)))
valid_df.tail()

[   18    46    51 ... 99781 99798 99804]


,sample_id,time_step,aileron_pos_lh_deg,aileron_pos_rh_deg,corrected_angle_of_attack_deg,baro_correct_alt_lsp_ft,computed_airspeed_lsp_knots,selected_course_deg,drift_angle_deg,elevator_pos_left_deg,...,total_pressure_lsp_millibar,pitch_angle_lsp_deg,roll_angle_lsp_deg,rudder_pos_deg,true_heading_lsp_deg,vertical_accel_g,wind_speed_knots,label,arrival_airport,arrival_airport_catenc
1590875,99804,155,87.95096,81.670350,-6.530688,563.91240,121.45615,172.87962,-1.551710,-4.686901,...,1017.48834,-2.657302,-0.318728,-3.936114,-161.69504,1.005220,8.503565,3.0,KAUS,NaN
1590876,99804,156,83.00012,81.036156,-4.481043,555.51050,119.55507,172.87962,-1.333551,-3.786747,...,1017.27734,-1.985865,-0.077250,-3.493000,-194.83679,1.001449,7.306917,3.0,KAUS,NaN
1590877,99804,157,88.64653,85.700580,-3.257945,545.90230,118.49332,172.87962,-0.874409,-4.359573,...,1016.63530,-1.002879,1.132922,-3.442693,137.02766,1.067352,7.246168,3.0,KAUS,NaN
1590878,99804,158,81.69081,77.558300,-2.722166,535.87260,114.45854,172.87962,0.227376,-2.456978,...,1015.44696,-0.953609,0.270099,-3.699163,182.29773,0.983827,4.381433,3.0,KAUS,NaN
1590879,99804,159,83.61386,81.036156,-1.036608,526.67505,114.67585,172.87962,0.818442,-2.988888,...,1016.45337,0.230453,0.618528,-3.912867,180.64206,1.100833,5.349127,3.0,KAUS,NaN


In [109]:
arrival_airport_mapping_dict = {airport: idx for idx, airport in enumerate(arrival_airport_mapping)}
valid_df['arrival_airport_catenc'] = valid_df['arrival_airport'].map(arrival_airport_mapping_dict)
valid_df['arrival_airport_catenc'] = valid_df['arrival_airport_catenc'].astype(float)
valid_df.drop(columns=['arrival_airport'], inplace=True)
valid_df.head()

,sample_id,time_step,aileron_pos_lh_deg,aileron_pos_rh_deg,corrected_angle_of_attack_deg,baro_correct_alt_lsp_ft,computed_airspeed_lsp_knots,selected_course_deg,drift_angle_deg,elevator_pos_left_deg,...,core_speed_avg_perc,total_pressure_lsp_millibar,pitch_angle_lsp_deg,roll_angle_lsp_deg,rudder_pos_deg,true_heading_lsp_deg,vertical_accel_g,wind_speed_knots,label,arrival_airport_catenc
0,18,0,83.184250,83.67523,-6.792032,1612.6204,128.703580,-68.026855,-6.184055,-6.569035,...,82.513760,977.36970,0.429042,20.637896,-0.185368,-138.77364,0.565935,16.215496,0.0,11.0
1,18,1,82.897835,84.22760,-6.425116,1613.3372,126.980020,-68.026855,-6.091631,-6.303082,...,82.715600,976.66974,0.300326,19.733183,-0.274315,-137.08560,0.705978,15.904416,0.0,11.0
2,18,2,82.877380,84.18669,-5.949559,1610.4536,126.013230,-68.026855,-6.332006,-7.735142,...,82.386680,976.38824,-0.242279,17.884089,-1.447740,-133.74431,0.517630,15.956974,0.0,11.0
3,18,3,82.856920,83.98210,-6.675802,1607.0038,125.520970,-68.026855,-6.655682,-5.955296,...,82.649414,976.35790,-1.455954,15.929121,-1.553204,-132.06366,0.672479,15.911472,0.0,11.0
4,18,4,82.877380,84.10486,-5.960420,1601.6227,125.407166,-68.026855,-7.219427,-5.750717,...,82.341070,976.48470,-1.506611,14.173342,-1.781981,-128.92786,0.071948,16.908192,0.0,11.0


In [110]:
valid_scaled = valid_df.copy()
for i in range (0,len(float_columns)):
  valid_scaled[float_columns[i]] = scalers[i].transform(valid_scaled[[float_columns[i]]])

In [111]:
valid_x = valid_scaled.copy()
valid_x = valid_x.drop(columns=['label','sample_id'])
valid_y = valid_scaled['label']

In [112]:
num_samples = len(train_x) // 160
num_features = train_x.shape[1]

# Reshape the data
train_data = train_x.values.reshape((num_samples, 160, num_features))
train_y = df_scaled['label']
train_y = np.array(train_y)
train_data_y = []
for i in range(0,len(train_y),160):
  train_data_y.append(train_y[i])
train_y = train_y.astype(int)
train_y = np.array(train_data_y)
print(train_y)

[0. 0. 0. ... 3. 3. 3.]


In [113]:
train_y = to_categorical(train_y)

In [114]:
# Build the model
model = Sequential()
model.add(LSTM(32, input_shape=(160, num_features), return_sequences=True) )  # First LSTM layer
model.add(Dropout(0.2))                    # Adding dropout for regularization
model.add(LSTM(16))                        # Second LSTM layer
model.add(Dense(8, activation='relu'))     # Additional Dense layer
model.add(Dense(4, activation='sigmoid'))  # Output layer

In [115]:
print(model.summary())

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_12 (LSTM)              (None, 160, 32)           7168      
                                                                 
 dropout_6 (Dropout)         (None, 160, 32)           0         
                                                                 
 lstm_13 (LSTM)              (None, 16)                3136      
                                                                 
 dense_8 (Dense)             (None, 8)                 136       
                                                                 
 dense_9 (Dense)             (None, 4)                 36        
                                                                 
Total params: 10476 (40.92 KB)
Trainable params: 10476 (40.92 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [116]:
# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [117]:
print("train_data type:", type(train_data))
print("train_y type:", type(train_y))

print("train_data dtype:", train_data.dtype if isinstance(train_data, np.ndarray) else "Not NumPy")
print("train_y dtype:", train_y.dtype if isinstance(train_y, np.ndarray) else "Not NumPy")

train_data type: <class 'numpy.ndarray'>
train_y type: <class 'numpy.ndarray'>
train_data dtype: object
train_y dtype: float32


In [118]:
hist = model.fit(train_data, train_y, epochs=175, batch_size=64)

ValueError: Failed to convert a NumPy array to a Tensor (Unsupported object type float).